In [59]:
import os
import numpy as np
import pandas as pd
import pydhn
from pydhn.fluids import Fluid
from pydhn.soils import Soil
from pydhn.solving import SimpleStep

# Parameters

In [60]:
t_supply_producer = 70 # °C
mdot_consumers = 553 / 3600 #kg/s
delta_t_consumers = - 30 # K

rho_pipe = 940 # kg/m3
lambda_pipe = 0.35 # W/kg/k
cp_pipe = 2000 # J/kg/K
roughness_pipe = 0.007 # mm

rho_ins = 50 # kg/m3
lambda_ins = 0.026 # W/kg/k
cp_ins = 1500 # J/kg/K

# Network creation

In [61]:

def create_net():
    # Load the files with the DESTEST data
    node_data = pd.read_csv("InputData/nodes_data.csv", sep=';')
    pipe_data = pd.read_csv("InputData/pipes_data.csv", sep=';')

    # Create network
    net = pydhn.Network()

    # Add nodes
    for i in node_data.index:
        row = node_data.loc[i]
        name = row["node_id"]
        name_s = name + "_s"
        name_r = name + "_r"
        X = row["x"]
        Y = row["y"]
        net.add_node(name=name_s, x=X, y=Y)
        net.add_node(name=name_r, x=X, y=Y)
        # Add consumers
        if "SimpleDistrict" in name:
            net.add_consumer(
                name=name,
                start_node=name_s,
                end_node=name_r,
                control_type="mass_flow",
                setpoint_type_hyd="mass_flow",
                setpoint_value_hyd=mdot_consumers,
                setpoint_value_hx=delta_t_consumers,
            )
        # Add producers
        if name == "i":
            net.add_producer(
                name="producer",
                start_node=name_r,
                end_node=name_s,
                setpoint_type_hx="t_out",
                setpoint_value_hx=t_supply_producer,
                setpoint_type_hyd="pressure",
                setpoint_value_hyd=-10000,
                static_pressure=100000,
            )

    # Add pipes
    for i in pipe_data.index:
        row = pipe_data.loc[i]
        start = row["start_node"]
        end = row["end_node"]
        name_s = f"{start}_to_{end}_s"
        name_r = f"{start}_to_{end}_r"
        start_s = f"{start}_s"
        end_s = f"{end}_s"
        start_r = f"{end}_r"
        end_r = f"{start}_r"
        length = row["length_m"]
        diameter = row["diameter_m"]
        thickness_ins = row["t_ins_m"]
        thickness_pipe = row["t_pipe_m"]

        net.add_pipe(
            name=name_s,
            start_node=start_s,
            end_node=end_s,
            diameter=diameter,
            length=length,
            roughness=roughness_pipe,
            k_insulation=lambda_ins,
            insulation_thickness=thickness_ins,
            k_internal_pipe=lambda_pipe,
            internal_pipe_thickness=thickness_pipe,
            casing_thickness=0.0,
            depth=1.5,
            line="supply",
        )
        net.add_pipe(
            name=name_r,
            start_node=start_r,
            end_node=end_r,
            diameter=diameter,
            length=length,
            roughness=roughness_pipe,
            k_insulation=lambda_ins,
            insulation_thickness=thickness_ins,
            k_internal_pipe=lambda_pipe,
            internal_pipe_thickness=thickness_pipe,
            casing_thickness=0.0,
            depth=1.5,
            line="return",
        )

    return net

# Simulation

In [62]:
def main():
    """
    Main function to run the simulation and save the results as a CSV file
    """
    # Create net, fluid and soil
    net = create_net()
    fluid = Fluid(name="Water", isconstant=True, cp=4180, mu=0.0005434, rho=988, k=0.64)
    soil = Soil(k=0.0, temp=10)

    # Run simulation
    hyd_kwargs = {"error_threshold": 1}
    thermal_kwargs = {"error_threshold": 1e-10}
    loop = SimpleStep(
        hydraulic_sim_kwargs=hyd_kwargs, thermal_sim_kwargs=thermal_kwargs
    )

    import time

    start_time = time.perf_counter()

    res = loop.execute(net=net, fluid=fluid, soil=soil)

    end_time = time.perf_counter()
    sim_time = end_time - start_time

    print(f"Average simulation time: {sim_time:.2f} s")

    # Initialize results DataFrame
    columns = [
        'Time',
        'Mass flow rate supply i [kg_h]',
        'Pressure drop supply between i and e [Pa]',
        'Pressure drop return between a and i [Pa]',
        'Pressure drop return between i and h [Pa]',
        'Fluid temperature supply i [C]',
        'Fluid temperature supply h [C]',
        'Fluid temperature supply g [C]',
        'Fluid temperature supply f [C]',
        'Fluid temperature supply e [C]',
        'Fluid temperature supply SimpleDistrict_1 [C]',
        'Fluid temperature return i [C]',
        'Fluid temperature return h [C]',
        'Fluid temperature return g [C]',
        'Fluid temperature return f [C]',
        'Fluid temperature return e [C]',
        'Fluid temperature return SimpleDistrict_1 [C]',
        'Heat loss supply between i and h [W]',
        'Total heat load supplied by heat source [W]'
    ]

    results_df = pd.DataFrame(0.0, index=[0], columns=columns)
    results_df['Time'] = 0

    # Get results
    mdot_supply = res["edges"]["mass_flow"][0, net.producers_mask[0]] * 3600

    # Pressure
    nodes, pressure = net.nodes(data="pressure")
    node_pressure = dict(zip(nodes, pressure))

    print(node_pressure)

    dp_i_to_e_s = node_pressure["i_s"] - node_pressure["e_s"]
    dp_a_to_i_r = node_pressure["a_r"] - node_pressure["i_r"]
    dp_i_to_h_r = node_pressure["i_r"] - node_pressure["h_r"]

    # Temperature
    nodes, temperature = net.nodes(data="temperature")
    node_temperature = dict(zip(nodes, temperature))
    node_names = ["i", "h", "g", "f", "e", "SimpleDistrict_1"]
    node_list = [f"{n}_{l}" for l in ["s", "r"] for n in node_names]
    node_temps = [node_temperature[n] for n in node_list]

    # Energy
    edges, energy = net.edges(data="delta_q")
    edges = [(u, v) for (u, v) in edges]
    q_w = dict(zip(edges, energy))
    i_to_h_q_w = q_w[("i_s", "h_s")]
    supply_q_w = q_w[("i_r", "i_s")]

    # Store results in DataFrame
    results_df["Mass flow rate supply i [kg_h]"] = mdot_supply
    results_df["Pressure drop supply between i and e [Pa]"] = np.abs(dp_i_to_e_s)
    results_df["Pressure drop return between a and i [Pa]"] = np.abs(dp_a_to_i_r)
    results_df["Pressure drop return between i and h [Pa]"] = np.abs(dp_i_to_h_r)
    t_cols = [col for col in results_df.columns if "Fluid temperature" in col]
    for i, col in enumerate(t_cols):
        results_df[col] = node_temps[i]
    results_df["Heat loss supply between i and h [W]"] = np.abs(i_to_h_q_w)
    results_df["Total heat load supplied by heat source [W]"] = np.abs(supply_q_w)

    # Save to CSV
    return results_df

In [63]:
results_df = main()

base_path = os.getcwd()
results_folder = os.path.join(base_path, 'Results')
results_file = os.path.join(results_folder, "network_CE0_pydhn_results.csv")
results_df.to_csv(results_file, index=False)

print(f"Results for CE0 implemented in pyDHN saved in: {results_file}")

Loop started
Hydraulic simulation converged after 0 iterations with an error of 3.637978807091713e-12 Pa
Thermal simulation converged after 1 iterations with an error of 2.842170943040401e-14 °C
Loop completed
Average simulation time: 0.12 s
{'SimpleDistrict_1_s': 75110.0329999222, 'SimpleDistrict_1_r': 114889.9670000778, 'SimpleDistrict_2_s': 75110.0329999222, 'SimpleDistrict_2_r': 114889.9670000778, 'SimpleDistrict_3_s': 75110.0329999222, 'SimpleDistrict_3_r': 114889.9670000778, 'SimpleDistrict_4_s': 75110.0329999222, 'SimpleDistrict_4_r': 114889.9670000778, 'SimpleDistrict_5_s': 78779.35064875602, 'SimpleDistrict_5_r': 111220.64935124398, 'SimpleDistrict_6_s': 78779.35064875602, 'SimpleDistrict_6_r': 111220.64935124398, 'SimpleDistrict_7_s': 78779.35064875602, 'SimpleDistrict_7_r': 111220.64935124398, 'SimpleDistrict_8_s': 78779.35064875602, 'SimpleDistrict_8_r': 111220.64935124398, 'SimpleDistrict_9_s': 83200.39750809039, 'SimpleDistrict_9_r': 106799.60249190961, 'SimpleDistrict_10

C:\Users\sara.ferrero\PycharmProjects\PandaPipesModel\.venv\Lib\site-packages\pydhn\solving\loops.py:168: UserWarning: Running a thermal simulation without a ts_id specified. This can lead to unexpected behaviours in dynamic components. To suppress this warning, pass a ts_id value when calling .execute()
  warn(
